# 05. Prototipas `predict_case` ir AI naudojimo žurnalas

> **Peržiūros notebook.** Šaltiniai: `app/predict_case.m`, `reports/ai_usage.md`, `hypotheses.csv`. Šis failas **nevykdo** prognozės.


## Kas čia vyksta paprastai

Po vieno `evaluate_test` galima paduoti **vieną** 1×30 atvejį ir gauti: P(M), slenkstį, sprendimą B/M, ar tikimybė arti slenksčio (`margin_flag`), ar požymiai išeina už mokymo min–max (`ood_flag`), modelio versiją ir atsakomybės sakinį. Žodžio „diagnozė“ kode nėra, išskyrus privalomą disclaimer. **Fit čia nedaromas** — tik užšaldytas `.mat`.


## `app/predict_case.m`

```matlab
% app/predict_case.m (fragmentas)
if exist(lockPath, 'file') ~= 2
    error('predict_case: testas dar neatrakintas — pirma evaluate_test.m.');
end
[modelName, tName] = choose_frozen_model(cfg);
Z = apply_scaler(xUsed, model.scaler);
[p, ~, yhat] = predict_proba(model, Z, t);
ood_flag = any(xUsed < xmin - 1e-12 | xUsed > xmax + 1e-12);
margin_flag = abs(p - t) < 0.10;
% disclaimer: mokomasis prototipas, ne klinikinė diagnozė
```

Įvestis: vektorius, struct, lentelė arba CSV kelias; NaN/Inf ir X<0 — klaida, ne imputacija.


## Kuris modelis prototipe?

Preregistracija: jei H1 atmetama, prototipas = **logistinė regresija** (arba **tiesinis SVM**, jei A2 geresnis).

| šaltinis | faktas |
|---|---|
| `hypotheses.csv` eilutė `H1` | `accepted=0` |
| `ablation.csv` A2 | `svm_linear` AUC 0,9953 > RBF 0,9863; Sp linear 0,887 < RBF 0,930; ΔSp PI kerta 0 |
| `main_results.csv` `t_se98` | LR Sp 0,5634; linear Sp 0,8873 |
| egzamino rašto taisyklė | prototipas = **LR** |
| kodas `choose_frozen_model` | po H1 atmetimo gali rinkti `svm_linear@t_se98`, jei linear Sp ≥ LR Sp |

Abu keliai užfiksuoti ataskaitoje; CSV nekeičiami. Tai dokumentacijos faktas, ne naujas eksperimentas.


## Kalibracija ir ribiniai atvejai prototipui

![Kalibracija](../reports/figures/calibration.png)

**Parašas.** Prototipo `p_malignant` yra Platt (ar LR) tikimybė. H2 priimta (`hypotheses.csv`), bet n = 113, nuolydžio PI platus — `margin_flag` (`|p−t|<0,10`) yra atsargos signalas, ne antras modelis.

Iš 04: FP `abs_idx=466` turi |p−t|=0,045 → `margin_flag` būtų 1. FN 41 p=0,048 — prototipas sakyti M **nepradėtų**, nepriklausomai nuo 0,10 juostos aplink t=0,34.

![SVM-RBF painiava](../reports/figures/confusion_svm_rbf.png)

**Parašas.** Tie patys 2 FN / 5 FP, kuriuos prototipas paveldėtų, jei veiktų SVM-RBF @ `t_se98`.


## AI naudojimo žurnalas (`reports/ai_usage.md`)

Taisyklė: AI citata nėra įrodymas. Priimta tik tai, kas sutikrinta su planu, MATLAB išvestimi arba CSV.

| Užklausa AI | Priimta / Atmesta | Priežastis |
|---|---|---|
| Visas 5 etapų MATLAB vamzdis pagal planą | Priimta (su korekcijomis) | Vardai = lentelės; `run_all`, testai, freeze + lock |
| Slenkstį / HP rinkti pagal **testo** Se/Sp/AUC | **Atmesta** | Nutekėjimas; lock draudžia antrą `evaluate_test` |
| `fit_scaler` ant visos 569×30 | Atmesta | `n=569` klaida; μ tik iš 456 |
| HoldOut = 114 / 455 | Atmesta kaip „taisymas“ | MATLAB davė **113/456**; priimta, nekirpta |
| Atmesti eilutes su nuliais concavity | Atmesta | Oficialiame WDBC nulių yra |
| HP „pirmas paprastas tinklelyje“ | **Atmesta (AI klaida #1)** | Žr. žemiau |
| Formulių numeriai (6)–(24) iš 25 p. plano | **Atmesta (AI klaida #2)** | Kolokviumas (1)–(9) |
| `t_cost`≈`t_se98` = programavimo klaida | Atmesta | Tas pats min EC OOF |
| H1 „SVM laimėjo, nes ΔSp didelis“ | Atmesta | `H1 accepted=0` dėl AUC PI |
| Prototipas = linear SVM (A2) | Kode 4.1; egzaminas sako LR | Abu keliai ataskaitoje |
| App Designer `.mlapp` | Atmesta (neprivaloma) | Paliktas `predict_case.m` |
| H3 keičia H1 | Atmesta | H3 po testo; H1 eilutė nepaliesta |


### AI klaida #1 — HP taisyklė

Pirminis `tune_cv` ėjo tinklu ir galėjo palikti **paprastesnę konfigūraciją su žemesniu AUC nei maksimumas**. Taisyklė pakeista: pirma visas tinklas, tada tarp AUC ≥ max−0,002 — paprastesnis. Patikra **prieš** `evaluate_test`. Testo metrikos HP nekeičia.

### AI klaida #2 — formulių numeriai

Komentaruose buvo plano (6)–(24). Kolokviumas: (1) z-score, (2)–(3) RBF/SVM, (4) Platt, (5)–(6) t ir EC, (7) Se/Sp, (8) AUC, (9) Brier. Perrašyta prieš testo atrakinimą. CSV skaičių tai nekeičia, bet „formulė ↔ kodas“ būtų buvęs klaidingas.

### Implementacijos lūžis

`error_analysis` antraštėje dubliuotas `area_worst` — `cell2table` klaida. Pataisyta, kol `test_unlocked.flag` dar neegzistavo; tai **ne** antras atrakintas testas.


## Notebook'ų žemėlapis

| Failas | Etapai |
|---|---|
| [01_duomenu_gavimas_ir_paruosimas.ipynb](01_duomenu_gavimas_ir_paruosimas.ipynb) | UCI, invariantai, 456/113, 25 CV, z-score be nutekėjimo |
| [02_modeliu_mokymas.ipynb](02_modeliu_mokymas.ipynb) | B0–M3, `tune_cv`, Platt, slenksčiai, freeze |
| [03_vertinimas_ir_hipotezes.ipynb](03_vertinimas_ir_hipotezes.ipynb) | vienas testas, `main_results`, H1–H3, grafikai |
| [04_abliacija_ir_klaidu_analize.ipynb](04_abliacija_ir_klaidu_analize.ipynb) | A0/A2/A5/A7, FN/FP |
| 05 (šis) | `predict_case`, AI žurnalas |

Peržiūra: atidaryti `.ipynb` Jupyter / VS Code / [nbviewer](https://nbviewer.org/) / Colab **be paleidimo**. Santykiniai keliai `../reports/figures/*.png` veikia, kai notebook'as guli `notebooks/` šalia `reports/`.
